# 04 — Generation Demo

Load a trained checkpoint and generate text from custom prompts.
Compare outputs across different temperatures and top-k values.

In [1]:
import sys
sys.path.insert(0, '..')

import torch
import tiktoken

from src.infra.config import load_config
from src.infra.io import load_checkpoint
from src.models.gpt.model import GPT
from src.core.generation import generate

In [2]:
# ── Load config and checkpoint ────────────────────────────────────────────
config = load_config('../configs/experiments/exp_001_baseline.yaml')
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = GPT(config.model).to(device)
ckpt_path = '../outputs/checkpoints/best_model.pt'
load_checkpoint(ckpt_path, model, device=device)
model.eval()

enc = tiktoken.get_encoding('gpt2')
print(f'Model loaded on {device}')

FileNotFoundError: Checkpoint not found: ../outputs/checkpoints/best_model.pt

In [3]:
# ── Generate text ─────────────────────────────────────────────────────────
def generate_text(prompt: str, max_tokens: int = 200,
                  temperature: float = 1.0, top_k: int = 50) -> str:
    tokens = enc.encode_ordinary(prompt)
    ctx = torch.tensor(tokens, dtype=torch.long, device=device).unsqueeze(0)
    out = generate(model, ctx, max_new_tokens=max_tokens,
                   temperature=temperature, top_k=top_k)
    return enc.decode(out.squeeze().tolist())

prompts = [
    'Once upon a time there was a pumpkin.',
    'A little girl went to the woods',
]
for p in prompts:
    print(f'--- Prompt: "{p}" ---')
    print(generate_text(p, max_tokens=200))
    print()

--- Prompt: "Once upon a time there was a pumpkin." ---


NameError: name 'enc' is not defined

In [4]:
# ── Temperature sweep ─────────────────────────────────────────────────────
prompt = 'Once upon a time'
for temp in [0.5, 0.8, 1.0, 1.2]:
    print(f'--- temperature={temp} ---')
    print(generate_text(prompt, max_tokens=80, temperature=temp, top_k=50))
    print()

--- temperature=0.5 ---


NameError: name 'enc' is not defined